In [1]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from stopit import threading_timeoutable as timeoutable
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)

eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
fig_DIR = "../figures/figures_fit/"
res_DIR = "../data/results_fit/"
cyc_DIR = "../data/cycling/"
# %matplotlib widget

In [2]:
parameter_values = get_parameter_values()

In [3]:
# parameter_values.search("Li")

In [4]:
spm = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
        # "loss of active material": ("stress-driven","none"),
        "loss of active material": "stress-driven",
        "lithium plating": "irreversible",
        "stress-induced diffusion": "false",
    }
)
# spm.print_parameter_info()
param=spm.param

In [5]:
cell = 9

## Load eSOH Data and OCV Data

In [6]:
cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe_0,spm,parameter_values)

In [7]:
pybamm.set_logging_level("WARNING")
# pybamm.set_logging_level("NOTICE")
# drive_cycle = pd.read_csv(cyc_DIR+'peyman_drive_cycle_current'+'.csv', comment="#", header=None).to_numpy()
experiment = pybamm.Experiment(
    [
        ("Discharge at "+c_rate_d+dis_set,
         "Rest for 5 min",
         "Charge at "+c_rate_c+" until 4.2V", 
         "Hold at 4.2V until C/100")
    ] *dfe_0.N.iloc[-1],
    termination="50% capacity",
#     cccv_handling="ode",
)

In [8]:
ic=4
blam_p = [4.0312e-08,4.0312e-08,4.0312e-08*5,5.7626e-08,5.6076e-08]
blam_n = [1.8157e-07,0.5*1.8157e-07,2*1.8157e-07*5,6.7490e-07,6.7429e-07]
blam_n2 = [4.9170e-09,0.5*4.9170e-09,2*4.9170e-09,1.7360e-07,1.7447e-07]
blam_p2 = [1.4406e-09,1.4406e-09,1.4406e-09,2.4735e-08,2.5257e-08]
k_pl = [2.3586e-09,2.3586e-09,2.3586e-09,1.4749e-08,2.3586e-09] 
m_lam= [1.0776,1.0776,1.0776,1.02,1.02]
x0 = np.array([1.0,1.0,1.0,1.0,-1,-1,1.0])

In [9]:
Temp

45

In [10]:
parameter_values = get_parameter_values()
parameter_values.update(
    {
        "Negative electrode active material volume fraction": eps_n_data,
        "Positive electrode active material volume fraction": eps_p_data,
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
        # "Positive electrode LAM constant proportional term [s-1]": 1.27152e-07,
        # "Negative electrode LAM constant proportional term [s-1]": 1.27272e-06,
        # "Positive electrode LAM constant exponential term": 1.1992,
        # "Negative electrode LAM constant exponential term": 1.1992,
        "SEI kinetic rate constant [m.s-1]":  x0[6]*4.6079e-16, #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]": 4.56607447e-19,#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
        "Initial inner SEI thickness [m]": 0e-09,
        "Initial outer SEI thickness [m]": 5e-09,
        "SEI resistivity [Ohm.m]": 30000.0,
        "Lithium plating kinetic rate constant [m.s-1]": x0[3]*k_pl[ic],
        # "Li plating resistivity [Ohm.m]": 0,
        "Li plating resistivity [Ohm.m]": 30000.0,
        "Positive electrode LAM constant proportional term [s-1]": x0[0]*blam_p[ic],
        "Negative electrode LAM constant proportional term [s-1]": x0[1]*blam_n[ic],
        "Positive electrode LAM constant exponential term": x0[2]*m_lam[ic],
        "Negative electrode LAM constant exponential term": x0[2]*m_lam[ic],
        
        "Positive electrode LAM constant proportional term 2 [s-1]": x0[5]*blam_p2[ic],
        "Negative electrode LAM constant proportional term 2 [s-1]": x0[4]*blam_n2[ic],
        "Negative electrode diffusion coefficient [m2.s-1]":8e-14,
        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
        "Negative electrode LAM min stress [Pa]": 0,
        "Negative electrode LAM max stress [Pa]": 0,
        "Positive electrode LAM min stress [Pa]": 0,
        "Positive electrode LAM max stress [Pa]": 0,
        # "Negative electrode critical stress [Pa]": 20e+06,
        # "Positive electrode critical stress [Pa]": 40e+06,
    },
    check_already_exists=False,
)

In [11]:
# all_sumvars_dict = cycle_adaptive_simulation_V2(spm, parameter_values, experiment,SOC_0, save_at_cycles=1)

In [12]:
# fig = plotc(all_sumvars_dict,dfe_0);
# fig.savefig(fig_DIR +'fast_sim_'+cell_no+'_new.png')

In [13]:
# fgdfdfg

# Parameter Fitting

## List of Initial Conditions

In [14]:
ic=4
blam_p = [4.0312e-08,4.0312e-08,4.0312e-08*5,5.7626e-08,5.6076e-08]
blam_n = [1.8157e-07,0.5*1.8157e-07,2*1.8157e-07*5,6.7490e-07,6.7429e-07]
blam_n2 = [4.9170e-09,0.5*4.9170e-09,2*4.9170e-09,1.7360e-07,1.7447e-07]
blam_p2 = [1.4406e-09,1.4406e-09,1.4406e-09,2.4735e-08,2.5257e-08]
k_pl = [2.3586e-09,2.3586e-09,2.3586e-09,1.4749e-08,2.3586e-09] 
m_lam= [1.0776,1.0776,1.0776,1.02]
x0 = np.array([1.0,1.0,1.0,1.0,-1,-1])

In [15]:
def objective(model, data):
    return np.array(model.loc[data['N_mod']]["Capacity [A.h]"]) - np.array(data["Capacity [A.h]"])

def multi_objective(model, data):
    # variables = ["C_n","C_p","x_100","y_0"]
    # weights = [1,1,5,5]
    # variables = ["Capacity [A.h]", "Loss of lithium inventory [%]"]
    # # weights = [1,1/20]
    variables = ["Capacity [A.h]", "Loss of lithium inventory [%]", "C_n", "C_p"]
    weights = [1,1/20,1,1]
    return np.concatenate([
        (np.array(model.loc[data['N_mod']][var]) - np.array(data[var])) * w
        for w,var in zip(weights,variables)
    ]
    )
@timeoutable()
def simulate(x,eps_n_data,eps_p_data,SOC_0,Temp,experiment,parameter_values):
    # simulate
    return cycle_adaptive_simulation_V2(spm, parameter_values, experiment, SOC_0,save_at_cycles=1,drive_cycle=None)
def prediction_error(x):
    cells = [3,9,12]
    try:
        out=[]
        for cell in cells:
            cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
            eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe_0,spm,parameter_values)
            # print(f"Cell: {cell_no}")
            experiment = pybamm.Experiment(
                [
                    ("Discharge at "+c_rate_d+dis_set,
                    "Rest for 5 min",
                    "Charge at "+c_rate_c+" until 4.2V", 
                    "Hold at 4.2V until C/50")
                ] *dfe_0.N.iloc[-1],
                termination="50% capacity",
            #     cccv_handling="ode",
            )
            parameter_values.update(
                {
                    "Positive electrode LAM constant proportional term [s-1]": x[0]*blam_p[ic],
                    "Negative electrode LAM constant proportional term [s-1]": x[1]*blam_n[ic],
                    "Positive electrode LAM constant exponential term": x[2]*2,
                    "Negative electrode LAM constant exponential term": x[2]*2,
                    # "SEI kinetic rate constant [m.s-1]":  x[3]*4.6079e-16,
                    "SEI kinetic rate constant [m.s-1]": 1.4840e-15,
                    "Lithium plating kinetic rate constant [m.s-1]": x[3]*k_pl[ic],
                    "Positive electrode LAM constant proportional term 2 [s-1]": x[5]*blam_p2[ic],
                    "Negative electrode LAM constant proportional term 2 [s-1]": x[4]*blam_n2[ic],
                    "Negative electrode active material volume fraction": eps_n_data,
                    "Positive electrode active material volume fraction": eps_p_data,
                    "Negative electrode diffusion coefficient [m2.s-1]":8e-14,
                    "Initial temperature [K]": 273.15+Temp,
                    "Ambient temperature [K]": 273.15+Temp,
                    "Negative electrode LAM min stress [Pa]": 0,
                    "Negative electrode LAM max stress [Pa]": 0,
                    "Positive electrode LAM min stress [Pa]": 0,
                    "Positive electrode LAM max stress [Pa]": 0,
                    "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
                },
                check_already_exists=False,
            )
            if cell == 15 or cell == 18:
               parameter_values.update(
                    {
                        "Negative electrode partial molar volume [m3.mol-1]": 7e-06,

                    },
                    check_already_exists=False,
                ) 
            model = cycle_adaptive_simulation_V2(spm, parameter_values, experiment, SOC_0,save_at_cycles=1,drive_cycle=None)
            out_t =   1/len(dfe_0)*multi_objective(pd.DataFrame(model), dfe_0)
            out=np.concatenate([out,out_t])
        print(f"x={x}, norm={np.linalg.norm(out)}")
    except Exception as error:
        out=[]
        for cell in cells:
            cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
            out_t = 1/len(dfe_0)*np.concatenate([np.array(dfe_0['Cap'])]*4)
            out=np.concatenate([out, out_t])
        out = 2*np.ones_like(out)
        print(f"Error")
        print(error)
        print(f"x={x}, norm={np.linalg.norm(out)}")
    return out

def train_model():
    timer = pybamm.Timer()
    x0 = np.array([1.0,1.0,1.0,1.0,-1,-1])
    x0 = np.array([1.0,1.0,0.51,1.0,-1.0,-1.0])
    lower = np.array([1e-1, 1e-1, 0.5, 1e-1, -1e+1, -1e+1])
    upper = np.array([1e+1, 1e+1, 1.50, 1e+2, -1e-1, -1e-1])
    dfo_opts = {
        "init.random_initial_directions":True,
        "init.run_in_parallel": True,
    }
    soln_dfols = dfols.solve(prediction_error, x0,bounds=(lower, upper), rhoend=1e-2, user_params=dfo_opts)
    print(timer.time())
    return soln_dfols
def sim_train(df):
    soln_dfols = train_model()
    xsol = soln_dfols.x
    df['x_0'][0]=round(xsol[0],4)*blam_p[ic]
    df['x_1'][0]=round(xsol[1],4)*blam_n[ic]
    df['x_2'][0]=round(xsol[2]*2,4)
    df['x_3'][0]=round(xsol[3],4)*k_pl[ic]
    # df['x_4'][0]=round(xsol[4],4)*alam_p[ic]
    df['x_4'][0]=round(xsol[4],4)*blam_n2[ic]
    df['x_5'][0]=round(xsol[5],4)*blam_p2[ic]
    df['obj'][0]=soln_dfols.f
    return xsol,df

In [16]:
df_x = pd.DataFrame(columns=['x_0','x_1','x_2','x_3','x_4','x_5','obj'], index=[0])

In [17]:
x,df_x = sim_train(df_x)

x=[ 1.    1.    0.51  1.   -1.   -1.  ], norm=0.17028935894712394
x=[ 0.96616127  0.99185382  0.5         1.00924175 -0.98343456 -1.00299245], norm=0.16330803257106266
x=[ 1.04714458  0.9738497   0.51255998  1.06688465 -0.95022176 -0.98835664], norm=0.18357072858182816
x=[ 1.00225344  1.00754296  0.5         0.99925981 -0.99702867 -0.99843581], norm=0.15772285845597078
x=[ 1.00623577  0.98110772  0.5         0.98422493 -0.99393291 -1.00080053], norm=0.165101050700687
x=[ 1.00299569  0.99603181  0.5         1.00915329 -1.01121569 -1.02347409], norm=0.1597969205351015
x=[ 0.99759285  0.99143914  0.5         1.00833259 -1.01679996 -0.98332401], norm=0.1615415793277993
x=[ 1.01498212  1.1063133   0.5         1.00381351 -0.99469961 -1.00593515], norm=0.1339737118757785
x=[ 1.17747982  1.34479161  0.5         1.0402114  -0.73948562 -1.1072446 ], norm=0.0962273287107746
x=[ 2.11987914  1.32786579  0.5         2.32201988 -0.63827843 -0.97188889], norm=0.08079756091369532
x=[ 2.11115197  1.3310

# Versions
V3: Working version  
V4: Constraint by 1/2 and 2  
V5: 1/5 and 5  
V6: 1/5 and 5 except for beta2s  
V7: kpl=0, retune ksei  
V8: kpl=0, retune ksei, better initial guess      
V9: kpl=0, retune ksei, different cost function, better initial guess    
V10: set plating to zero  
V11: two stage: tune ksei with C/5 and then retune with plating  
V12: split tuning: tune ksei with C/5 and plating with 2C simultaneously  

In [18]:
sim_des="plating_mech_3_9_12_V11"
df_x.to_csv(res_DIR + "cycl_train_"+sim_des+".csv")

In [19]:
res_DIR

'../data/results_fit/'

In [20]:
sdsdsdsd

NameError: name 'sdsdsdsd' is not defined

In [ ]:
xsol = x

In [ ]:
print(f"{df_x['x_0'][0]:0.4e},{df_x['x_1'][0]:0.4e},{df_x['x_2'][0]},{df_x['x_3'][0]:0.4e},{df_x['x_4'][0]:0.4e},{df_x['x_5'][0]:0.4e}")

In [ ]:
def plotcn(all_sumvars_dict,esoh_data):
    esoh_vars = ["x_100", "y_0", "C_n_loss", "C_p_loss", "Capacity [A.h]", "Loss of lithium inventory [%]"]
    all_sumvars_dict["C_n_loss"] = (all_sumvars_dict["C_n"][0]-all_sumvars_dict["C_n"])/all_sumvars_dict["C_n"][0]*100
    all_sumvars_dict["C_p_loss"] = (all_sumvars_dict["C_p"][0]-all_sumvars_dict["C_p"])/all_sumvars_dict["C_p"][0]*100
    esoh_data["C_n_loss"] = (esoh_data["C_n"][0]-esoh_data["C_n"])/esoh_data["C_n"][0]*100
    esoh_data["C_p_loss"] = (esoh_data["C_p"][0]-esoh_data["C_p"])/esoh_data["C_p"][0]*100
    fig, axes = plt.subplots(3,2,figsize=(7,7))
    for k, name in enumerate(esoh_vars):
        ax = axes.flat[k]
        ax.plot(all_sumvars_dict["Cycle number"],all_sumvars_dict[name],"ro")
        ax.plot(esoh_data["N"],esoh_data[name],"kx")
        ax.set_title(split_long_string(name))
        # if k ==2 or k==3:
        #     ax.set_ylim([3,6.2])
        if k>3:
            ax.set_xlabel("Cycle number")
    fig.legend(["Sim"] + ["Data"], 
           loc="lower center",bbox_to_anchor=[0.5,-0.02], ncol=1, fontsize=11)
    fig.tight_layout()
    return fig

In [ ]:
sim_des

In [ ]:
fig_DIR